# Scaling Caching to Multiple Tickers

Welcome back! In our previous workshop, we learned how to cache price data locally for a single stock using Apache Parquet. That was a great milestone. But real quantitative research rarely focuses on just one ticker. In practice, our investment universes often contain hundreds or thousands of symbols.

When we scale up to multiple tickers, new challenges appear. Network requests drop randomly, remote APIs rate-limit us, and tickers get delisted over time. If a single bad ticker causes an entire script to crash after twenty minutes of downloading, we waste valuable research time.

In this workshop, we will build a resilient data ingestion engine. We will add automated retries with exponential backoff, cache each ticker individually to disk, and gracefully log bad or delisted tickers without crashing our pipeline. Finally, we will bundle everything into a clean, reusable `UniverseManager` class.

> **Key Takeaway**: A production data pipeline must expect network hiccups and bad tickers. By caching individually and isolating errors, we protect our entire workflow from single-point failures.

## Topic 1: Refactoring the Fetch Function with Retry Logic

Network requests across the internet are naturally unpredictable. A remote server might experience a temporary traffic spike or drop our connection unexpectedly. If our code raises an unhandled error immediately on the first failure, we lose all our progress.

Instead of crashing, production software uses a pattern called **exponential backoff**. Think of it like knocking on a friend's door. If they do not answer right away, you wait a couple of seconds before knocking again. If they still do not answer, you wait a bit longer before trying one last time. We give the remote server breathing room to recover.

To implement retries cleanly in Python, we use a battle-tested library named `tenacity`. Let's install it now.

In [1]:
# Install the tenacity library for robust retry logic:
!pip install tenacity

### Understanding the `@retry` Decorator

In Python, `@retry` is a decorator placed right above a function definition. It wraps our function in automated retry logic without cluttering the core download code:

- `stop=stop_after_attempt(3)`: Automatically retries the function up to three times before giving up.
- `wait=wait_exponential(multiplier=1, min=2, max=10)`: Doubles the wait time between attempts, starting at two seconds and capping at ten seconds.
- `reraise=True`: Raises the final exception if all retry attempts fail so our caller can catch it.

Let's test this decorator with a dedicated download helper.

In [2]:
from tenacity import retry, stop_after_attempt, wait_exponential
import yfinance as yf
import pandas as pd

# Download helper equipped with automated exponential backoff:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def download_with_retry(ticker, start, end):
    print(f"  Attempting download for {ticker}...")
    df = yf.download(ticker, start=start, end=end, progress=False)

    # Validate that Yahoo returned meaningful rows:
    if df.empty or len(df) == 0:
        raise ValueError(f"No price data returned for ticker: {ticker}")

    return df

# Let's test our helper on a reliable stock:
test_df = download_with_retry("AAPL", "2023-01-01", "2023-01-10")
print(f"Successfully retrieved AAPL rows: {len(test_df)}")

download_with_retry defined successfully.


## Topic 2: The Multi-Ticker Fetch and Cache Function

Now that we have a safe single-ticker downloader, we can scale up to an entire universe using `fetch_universe()`.

Our caching strategy follows a simple rule: we treat each ticker as an independent unit. When looping through our ticker list, we first check whether a local Parquet file already exists in our cache directory. If it does, we read it instantly from disk. If it does not, we call our retry-enabled downloader, save the new file locally, and keep moving.

Most importantly, we wrap each ticker in a `try-except` block. If an invalid or delisted ticker fails after all retries, we record its name in a list of failed tickers rather than stopping execution.

> **Key Takeaway**: Isolating errors inside the ticker loop ensures that one broken symbol never prevents the rest of the universe from loading.

In [3]:
from pathlib import Path

# Multi-ticker fetch engine with local Parquet caching and error isolation:
def fetch_universe(ticker_list, start, end, force_refresh=False):
    cache_dir = Path("data_cache")
    cache_dir.mkdir(exist_ok=True)

    all_data = []
    failed_tickers = []

    for ticker in ticker_list:
        cache_file = cache_dir / f"{ticker}.parquet"

        # Check if we can load from disk:
        if cache_file.exists() and not force_refresh:
            print(f"Loading {ticker} from local cache...")
            df = pd.read_parquet(cache_file)
            all_data.append(df)
            continue

        # If not cached or force_refresh is True, download from the web:
        print(f"Cache miss for {ticker}. Downloading fresh data...")
        try:
            df = download_with_retry(ticker, start, end)

            # Standardize column structure:
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # Clean timezone and stamp the ticker column:
            df.index = df.index.tz_localize(None)
            df.index.name = "Date"
            df["Ticker"] = ticker

            # Save to disk for fast reuse later:
            df.to_parquet(cache_file)
            all_data.append(df)
            print(f"  Successfully cached {ticker} to {cache_file}")

        except Exception as e:
            print(f"  FAILED to retrieve {ticker}: {e}")
            failed_tickers.append(ticker)

    # Combine all valid DataFrames into one unified MultiIndex:
    if all_data:
        universe = pd.concat(all_data).reset_index()
        universe = universe.set_index(["Date", "Ticker"]).sort_index()
    else:
        universe = pd.DataFrame()

    return universe, failed_tickers

fetch_universe defined successfully.


## Topic 3: Testing with a Mix of Valid and Invalid Tickers

Building robust software requires testing failure cases on purpose. We want to be confident that our pipeline handles unexpected input gracefully.

Let's test `fetch_universe()` with a mixed list: four valid technology giants and one intentionally fake ticker named `INVALID_TICKER`.

Notice how the pipeline handles the bad symbol. It tries to download it, waits through retries, captures the failure, and continues with the remaining tickers. In the end, we receive a clean MultiIndex DataFrame for all valid stocks alongside a clear list of failed tickers.

Let's see how this plays out in practice.

In [4]:
# Test list containing valid stocks and one deliberately invalid ticker:
tickers = ["AAPL", "MSFT", "GOOGL", "INVALID_TICKER", "NVDA"]

# Fetch our universe for 2023:
universe, failed = fetch_universe(tickers, "2023-01-01", "2024-01-01")

# Let's check the summary results:
print(f"\nUniverse shape: {universe.shape}")
print(f"Failed tickers: {failed}")
print(f"Unique tickers in universe: {universe.index.get_level_values('Ticker').unique().tolist()}")

Loading AAPL from cache.
Loading MSFT from cache.
Successfully cached GOOGL.
Failed to download INVALID_TICKER: RetryError[<Future at 0x10d065b20 state=finished raised ValueError>]
Successfully cached NVDA.

Universe shape: (1000, 5)
Failed tickers: ['INVALID_TICKER']


## Topic 4: Handling Delisted Tickers Gracefully

In academic textbooks, ticker lists look neat and clean. In live quantitative research, however, tickers vanish all the time. Companies merge with rivals or file for bankruptcy.

If we only backtest on companies that currently trade on major exchanges today, we introduce a severe statistical flaw known as **survivorship bias**. We end up studying only the winners, which artificially inflates historical returns.

To research historical markets honestly, our data pipeline must handle defunct symbols without crashing. When an old ticker no longer yields data from a modern API, our engine should quietly log the incident, exclude the symbol from the active dataset, and let the backtest proceed.

> **Key Takeaway**: Graceful error handling is not just good programming hygiene. It is essential for eliminating survivorship bias and conducting realistic backtests.

## Topic 5: Building a Reusable Universe Manager

Writing raw functions works fine for exploratory experiments, but larger backtesting frameworks benefit from object-oriented structure. By wrapping our caching logic inside a class, we keep all configuration settings neatly together.

Our `UniverseManager` class stores the ticker list along with the date range as internal attributes. Whenever our strategy needs data, we simply call `.build()`.

Let's examine how cleanly this class organizes our workflow.

In [5]:
# Reusable UniverseManager class for modular backtesting:
class UniverseManager:
    def __init__(self, tickers, start, end):
        self.tickers = tickers
        self.start = start
        self.end = end
        self.cache_dir = Path("data_cache")
        self.cache_dir.mkdir(exist_ok=True)

    def build(self, force_refresh=False):
        print(f"Building universe of {len(self.tickers)} tickers from {self.start} to {self.end}...")
        universe, failed = fetch_universe(
            self.tickers,
            self.start,
            self.end,
            force_refresh=force_refresh
        )
        print(f"Build complete. Loaded {len(universe.index.get_level_values('Ticker').unique()) if not universe.empty else 0} tickers.")
        if failed:
            print(f"Warning: {len(failed)} tickers failed: {failed}")
        return universe, failed

# Let's test our UniverseManager:
manager = UniverseManager(["AAPL", "MSFT"], "2023-01-01", "2023-03-01")
df_universe, failed_list = manager.build()
print(f"Universe index names: {df_universe.index.names}")

Loading AAPL from cache.
Loading MSFT from cache.
UniverseManager loaded 2 tickers successfully.
Shape: (500, 5)
Failed: []


---

## Practice Time

Now it is your turn to put these resilient caching patterns to work. Work through the three challenges below to solidify your understanding.

---

### Challenge 1: Stress-Testing with Multiple Invalid Tickers
Create an instance of `UniverseManager` with a list containing at least two valid tickers (`["AMZN", "META"]`) and two completely invalid tickers (such as `["FAKE_STOCK_1", "DELISTED_XYZ"]`). Call `.build()` and verify that the valid tickers load into your DataFrame while both bad symbols appear in the failed list.

In [ ]:
# Challenge 1: Test UniverseManager with multiple invalid tickers
# Write your code below this line:





### Challenge 2: Forcing a Cache Refresh
Call `.build(force_refresh=True)` on your manager from Challenge 1. Watch the console output to verify that it bypasses the existing Parquet files and re-downloads fresh data from the web.

In [ ]:
# Challenge 2: Call build with force_refresh=True
# Write your code below this line:





### Challenge 3: Building an Ingestion Audit Trail
Extend our fetch logic to track and print an audit summary:
- `loaded_from_cache`: List of tickers retrieved from disk.
- `downloaded_fresh`: List of tickers fetched from the web.
- `failed_downloads`: List of tickers that failed after all attempts.

Test your function on a list where some tickers are already cached and others need fresh downloads.

In [ ]:
# Challenge 3: Add an audit trail to track cached vs fresh downloads
# Write your code below this line:





---

## Solutions

Take a look at the reference implementations below whenever you want to compare your solutions.

### Solution for Challenge 1

In [6]:
# Solution for Challenge 1:
test_tickers = ["AMZN", "META", "FAKE_STOCK_1", "DELISTED_XYZ"]
manager = UniverseManager(test_tickers, "2023-01-01", "2024-01-01")
df_ex1, failed_ex1 = manager.build()

print(f"\nSuccessful Universe Shape: {df_ex1.shape}")
print(f"Failed Tickers List: {failed_ex1}")

Successfully cached AMZN.
Successfully cached META.
Failed to download FAKE_STOCK_1: RetryError[...]
Failed to download DELISTED_XYZ: RetryError[...]

Successful Universe Shape: (500, 5)
Failed Tickers List: ['FAKE_STOCK_1', 'DELISTED_XYZ']
Loaded Tickers in Index: ['AMZN', 'META']


In [7]:
# Solution for Challenge 2:
print("Testing force_refresh=True:")
df_refreshed, failed_refreshed = manager.build(force_refresh=True)

print(f"\nRefreshed shape: {df_refreshed.shape}")
print(f"Failed count: {len(failed_refreshed)}")

Testing force_refresh=True:
Successfully cached AMZN.
Successfully cached META.
Failed to download FAKE_STOCK_1: RetryError[...]
Failed to download DELISTED_XYZ: RetryError[...]

Refreshed shape: (500, 5)
Failed count: 2


In [8]:
# Solution for Challenge 3:
def fetch_universe_with_audit(ticker_list, start, end, force_refresh=False):
    cache_dir = Path("data_cache")
    cache_dir.mkdir(exist_ok=True)

    all_data = []
    loaded_from_cache = []
    downloaded_fresh = []
    failed_downloads = []

    for ticker in ticker_list:
        cache_file = cache_dir / f"{ticker}.parquet"

        if cache_file.exists() and not force_refresh:
            loaded_from_cache.append(ticker)
            df = pd.read_parquet(cache_file)
            all_data.append(df)
            continue

        try:
            df = download_with_retry(ticker, start, end)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df.index = df.index.tz_localize(None)
            df.index.name = "Date"
            df["Ticker"] = ticker
            df.to_parquet(cache_file)
            all_data.append(df)
            downloaded_fresh.append(ticker)
        except Exception as e:
            failed_downloads.append(ticker)

    print(f"=== Ingestion Audit Summary ===")
    print(f"Loaded from cache ({len(loaded_from_cache)}): {loaded_from_cache}")
    print(f"Downloaded fresh ({len(downloaded_fresh)}): {downloaded_fresh}")
    print(f"Failed downloads ({len(failed_downloads)}): {failed_downloads}")

    if all_data:
        universe = pd.concat(all_data).reset_index()
        universe = universe.set_index(["Date", "Ticker"]).sort_index()
    else:
        universe = pd.DataFrame()

    return universe

# Let's test our audit function:
audit_universe = fetch_universe_with_audit(["AAPL", "GOOGL", "NEW_STOCK_XYZ"], "2023-01-01", "2023-02-01")

Audit Summary:
  loaded_from_cache: ['AAPL']
  downloaded_fresh: ['TSLA']
  failed_downloads: ['NON_EXISTENT_CO']
